# Grid Search Hyperparameter HDBSCAN pada UMAP 30-dim

Mencari kombinasi mcs/ms terbaik untuk HDBSCAN setelah LOF+UMAP.
Input: umap_coords_inlier.npy dari NB10 (14.790 wajah, sudah LOF-filtered).

**Hasil terbaik:** mcs=15, ms=10 → DBCV=0.8391, Coverage=97.0%, n_clusters=123

In [ ]:
import numpy as np
import hdbscan
import pickle
import pandas as pd
from itertools import product

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

In [ ]:
X_umap = np.load("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb10/umap_coords_inlier.npy")

with open("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb10/skenario_d_results.pkl", "rb") as f:
    results_d = pickle.load(f)

print(f"X_umap shape : {X_umap.shape}")
print(f"Keys pkl     : {list(results_d.keys())}")

In [ ]:
MCS_LIST = [5, 10, 15, 25, 50]
MS_LIST  = [2, 5, 10]

rows = []
for mcs, ms in product(MCS_LIST, MS_LIST):
    if ms > mcs:
        continue

    clu = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=ms,
        metric='euclidean',
        cluster_selection_method='eom',
        gen_min_span_tree=False
    )
    labels = clu.fit_predict(X_umap)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = int((labels == -1).sum())
    coverage   = (labels >= 0).sum() / len(labels)

    if n_clusters >= 2:
        try:
            dbcv = hdbscan.validity_index(X_umap.astype(np.float64), labels)
        except Exception:
            dbcv = float('nan')
    else:
        dbcv = float('nan')

    rows.append(dict(mcs=mcs, ms=ms, n_clusters=n_clusters,
                     n_noise=n_noise, coverage_pct=round(coverage*100,1),
                     dbcv=round(dbcv,4)))
    print(f"mcs={mcs:3d} ms={ms:2d} → k={n_clusters:4d} noise={n_noise:4d} cov={coverage*100:.1f}% DBCV={dbcv:.4f}")

In [ ]:
df = pd.DataFrame(rows).sort_values('dbcv', ascending=False)
print(df.to_string(index=False))